# Experiment 10 (RQ3): Supervised Strands adaptation

One notebook for the four candidate adaptations of Strands 2B: calibration (RQ3A), the decision head (RQ3B), LoRA (RQ3C) and full adaptation (RQ3D). Design and qualification requirements only. No scientific training result is available; fit counts are not performance evidence.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.utils.config import load_yaml
from src.visualize import figures, style
from src.evaluation.research_questions import adaptation_plan, adaptation_prediction_plan
style.apply()
NOTEBOOK = "exp10_adaptation"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope and source identity")
display_json("Source", provenance())
record("No scientific improvement result is available here. Fit counts describe the design, not performance.", "Evidence boundary")
adaptation = pd.DataFrame(adaptation_plan())
record(load_yaml("exp10_adaptation/adaptation"), "Current design and pending runtime gates")

## 2. RQ3A: Calibration

In [2]:
report_section("2. RQ3A: Calibration design")
record({"fits":45,"training":"One scalar temperature per classification dataset and outer fold, fitted on a calibration subset inside outer training. No model weight training.","comparison":"Original probabilities versus calibrated probabilities on untouched outer test rows.","measures":["log loss","Brier","AUC","calibration bins"],"status":"not executed"}, "Calibration experiment")

## 3. RQ3B: Decision head

In [3]:
report_section("3. RQ3B: Decision head")
selected = adaptation[adaptation.variant.eq("head_only")]
display_table("Planned training fits", selected.groupby("variant").agg(training_runs=("fold","size"), pilot_runs=("pilot","sum")).reset_index())

variant,training_runs,pilot_runs
head_only,45,2


## 4. RQ3C: LoRA adaptation

In [4]:
report_section("4. RQ3C: LoRA adaptation")
selected = adaptation[adaptation.variant.eq("lora_and_head")]
display_table("Planned training fits", selected.groupby("variant").agg(training_runs=("fold","size"), pilot_runs=("pilot","sum")).reset_index())

variant,training_runs,pilot_runs
lora_and_head,45,2


## 5. RQ3D: Full adaptation

In [5]:
report_section("5. RQ3D: Full adaptation")
selected = adaptation[adaptation.variant.eq("full_backbone_and_head")]
display_table("Planned training fits", selected.groupby("variant").agg(training_runs=("fold","size"), pilot_runs=("pilot","sum")).reset_index())
record("Train the language backbone and decision head from the same released function. The publisher's adapter/head saver is not sufficient for changed backbone weights: full-weight serialization, optimizer/RNG resume, function-preserving initialization and B200 memory checks are required. This arm is designed but not implemented or enabled.", "Full-training gate")

variant,training_runs,pilot_runs
full_backbone_and_head,45,2


## 6. Shared per-field learner and held-out evaluation

Common to all four methods.

In [6]:
report_section("6. Shared per-field learner and held-out evaluation")
record("Split source rows into five outer folds before making text-field examples. Within each outer training set, pool its nonempty fields and their row class labels to fit one shared Strands model per method. Reuse that model for every text field and all six predictors. Keep missing feature blocks missing. Fixed 0.5 binary decisions avoid inner threshold fits, not outer tests. Optimizer, checkpoint-selection, temperature-calibration and test roles remain disjoint. RQ3 is classification-only; it does not reinterpret the nine regression evidence levels as target bins.", "Training boundary")
record({"planned_predictor_fits": len(adaptation_prediction_plan()), "includes_reusable_frozen_reference": True, "runner_implemented": False}, "Execution status")

## Complete report

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nOnly saved evidence was analysed; no inference, model training or live VSC changes were performed.")

exp10_adaptation

1. Scope and source identity


Source
{
  "code_sha256": "6593d29f281dcfc0d9eccf63a245c85c37280db4fed37cd9119be97f14ca2095",
  "git_head": "177427c62a24b29645898e7097943cd650c3b01f",
  "git_dirty": true,
  "tfm_library_pin": "1e5c8fe2c19d746fe3645c584df0c030bc03a117 dm-library (remotes/origin/HEAD)\n 60dffcdab88f20116fe0b17ac5aedfeb96aaa345 tfm-library (remotes/origin/HEAD)\n 211bc65a6155572fef808f2e7bd305c9395a441c vsc-library (heads/main-6-g211bc65)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Evidence boundary
No scientific improvement result is available here. Fit counts describe the design, not performance.


Current design and pending runtime gates
{
  "research_question": "Can supervised Strands adaptation improve per-field text features?",
  "status": "design_updated_training_runner_not_implemented",
  "enabled": false,
  "provider":